# LIBERO — **잘못된 eval(5000ep 등) 삭제, 500ep 만 남김**

eval 폴더를 `eval_info.json` 의 **에피소드 수**로 판단해서, **500ep 이 아닌 것(5000ep·50ep 등)** 을 지운다.
그러면 깔끔하게 500ep eval 만 남는다.

- **① 계획(dry-run)** 으로 뭐가 지워지고 뭐가 남는지 먼저 확인 → **② EXECUTE=True** 로 삭제 → **③ 확인**.
- eval_info 가 있는 **그 run 폴더**만 지운다(500ep 폴더는 그대로). 체크포인트(train)는 안 건드림.
- ⚠️ 삭제는 되돌릴 수 없으니 ① 목록 꼭 확인.


In [ ]:
import sys, json, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK    = 'libero_10'
KEEP_EP = 500                              # 이 ep 수만 남긴다. 나머지(5000·50 등)는 삭제.
EVAL_ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
print('scan:', EVAL_ROOT, '| exists:', EVAL_ROOT.is_dir())
print(f'남길 것 = {KEEP_EP}ep,  지울 것 = {KEEP_EP}ep 아닌 모든 eval')

## 1) 삭제/유지 계획 (dry-run)


In [ ]:
# ── 스캔 → 삭제/유지 계획 (dry-run: 아무것도 안 지움) ──
keep, delete = [], []
for info in sorted(EVAL_ROOT.rglob('eval_info.json')):
    try:
        ov = json.loads(info.read_text()).get('overall', {})
    except Exception:
        continue
    n_ep = ov.get('n_ep', ov.get('n_episodes')) or 0
    rel = info.parent.relative_to(EVAL_ROOT)
    (keep if n_ep == KEEP_EP else delete).append((rel, n_ep, info.parent))

print(f'■ 유지 ({KEEP_EP}ep) — {len(keep)}개:')
for rel, n, _ in keep:
    print(f'   {n:>6}ep   {rel}')
print(f'\n■ 삭제 ({KEEP_EP}ep 아님) — {len(delete)}개:')
for rel, n, _ in delete:
    print(f'   {n:>6}ep   {rel}   ← 삭제 대상')
if not delete:
    print('   (없음 — 전부 500ep, 깨끗함)')

## 2) 삭제 실행 (EXECUTE=True 로 바꿔 다시 실행)


In [ ]:
# ── 실제 삭제 ── 위 계획 확인했으면 EXECUTE=True 로 바꿔 다시 실행 ──
EXECUTE = False

if not delete:
    print('지울 것 없음.')
elif not EXECUTE:
    print(f'DRY-RUN — {len(delete)}개 삭제 예정(아직 안 지움). 위 목록 확인 후 EXECUTE=True 로 실행.')
else:
    done = 0
    for rel, n, p in delete:
        if Path(p).is_dir():
            shutil.rmtree(p); done += 1
            print(f'   삭제됨: {rel} ({n}ep)')
    print(f'\n완료: {done}개 삭제. 이제 {KEEP_EP}ep 만 남음.')

## 3) 확인 — 이제 500ep 만 남았나


In [ ]:
# ── 확인: 이제 뭐가 남았나 (model×seed 별 ep) ──
from collections import defaultdict
left = defaultdict(dict)
for info in sorted(EVAL_ROOT.rglob('eval_info.json')):
    try:
        ov = json.loads(info.read_text()).get('overall', {})
    except Exception:
        continue
    n_ep = ov.get('n_ep', ov.get('n_episodes')) or 0
    parts = info.parent.relative_to(EVAL_ROOT).parts
    model = parts[0] if parts else '?'
    seed = next((p for p in parts if p.startswith('seed')), '?')
    left[model][seed] = n_ep

print('남은 eval (model → seed:ep):')
bad = 0
for m in sorted(left):
    cells = '  '.join(f'{s}:{n}' for s, n in sorted(left[m].items()))
    print(f'   {m:<14} {cells}')
    bad += sum(1 for n in left[m].values() if n != KEEP_EP)
print(f'\n{KEEP_EP}ep 아닌 것 {bad}개' + (' — 아직 남음(EXECUTE 했나?)' if bad else ' ✅ 전부 500ep 깨끗'))